In [ ]:
Hands-on Lab: Getting Started with Apache Airflow

In [ ]:
Run the command below in the terminal to list all the existing DAGs.

airflow dags list

In [ ]:
Run the command below in the terminal to list all tasks in the DAG named example_bash_operator.

airflow tasks list example_bash_operator

In [ ]:
Run the command below in the terminal to unpause a DAG named tutorial.

airflow dags unpause tutorial

Run the command to pause the DAG.

airflow dags pause tutorial

In [ ]:
Hands-on Lab: Create a DAG for Apache Airflow with PythonOperator

In [ ]:
create this new file as a script → my_first_dag.py

# Import the libraries
from datetime import timedelta
# The DAG object; we'll need this to instantiate a DAG
from airflow.models import DAG
# Operators; you need this to write tasks!
from airflow.operators.python import PythonOperator

# This makes scheduling easy
from airflow.utils.dates import days_ago

# Define the path for the input and output files
input_file = '/etc/passwd'
extracted_file = 'extracted-data.txt'
transformed_file = 'transformed.txt'
output_file = 'data_for_analytics.csv'


def extract():
    global input_file
    print("Inside Extract")
    # Read the contents of the file into a string
    with open(input_file, 'r') as infile, \
            open(extracted_file, 'w') as outfile:
        for line in infile:
            fields = line.split(':')
            if len(fields) >= 6:
                field_1 = fields[0]
                field_3 = fields[2]
                field_6 = fields[5]
                outfile.write(field_1 + ":" + field_3 + ":" + field_6 + "\n")


def transform():
    global extracted_file, transformed_file
    print("Inside Transform")
    with open(extracted_file, 'r') as infile, \
            open(transformed_file, 'w') as outfile:
        for line in infile:
            processed_line = line.replace(':', ',')
            outfile.write(processed_line + '\n')


def load():
    global transformed_file, output_file
    print("Inside Load")
    # Save the array to a CSV file
    with open(transformed_file, 'r') as infile, \
            open(output_file, 'w') as outfile:
        for line in infile:
            outfile.write(line + '\n')


def check():
    global output_file
    print("Inside Check")
    # Save the array to a CSV file
    with open(output_file, 'r') as infile:
        for line in infile:
            print(line)


# You can override them on a per-task basis during operator initialization
default_args = {
    'owner': 'Your name',
    'start_date': days_ago(0),
    'email': ['your email'],
    'retries': 1,
    'retry_delay': timedelta(minutes=5),
}

# Define the DAG
dag = DAG(
    'my-first-python-etl-dag',
    default_args=default_args,
    description='My first DAG',
    schedule_interval=timedelta(days=1),
)

# Define the task named execute_extract to call the `extract` function
execute_extract = PythonOperator(
    task_id='extract',
    python_callable=extract,
    dag=dag,
)

# Define the task named execute_transform to call the `transform` function
execute_transform = PythonOperator(
    task_id='transform',
    python_callable=transform,
    dag=dag,
)

# Define the task named execute_load to call the `load` function
execute_load = PythonOperator(
    task_id='load',
    python_callable=load,
    dag=dag,
)

# Define the task named execute_load to call the `load` function
execute_check = PythonOperator(
    task_id='check',
    python_callable=check,
    dag=dag,
)

# Task pipeline
execute_extract >> execute_transform >> execute_load >> execute_check


In [ ]:
Exercise 4: Submit a DAG

Open a terminal and run the command below to set the AIRFLOW_HOME.

export AIRFLOW_HOME=/home/project/airflow
echo $AIRFLOW_HOME

Run the command below to submit the DAG that was created in the previous exercise.
 cp my_first_dag.py $AIRFLOW_HOME/dags

In [ ]:
Run the command below to list out all the existing DAGs.

airflow dags list

In [ ]:
Hands-on Lab: Create a DAG for Apache Airflow with BashOperator
In this lab, you will create workflows using BashOperator in Airflow DAGs and simulate an ETL process 
using bash commands that are scheduled to run once a day.

In [ ]:
Let's create a DAG that runs daily, and extracts user information from /etc/passwd file, transforms it, and loads it into a file.

This DAG will have two tasks extract that extracts fields from /etc/passwd file and transform_and_load that transforms and loads data into a file.

1- Create a new file by choosing File->New File and naming it my_first_dag.py.
2- Then, copy the code above and paste it into my_first_dag.py.

# import the libraries

from datetime import timedelta
# The DAG object; we'll need this to instantiate a DAG
from airflow.models import DAG
# Operators; you need this to write tasks!
from airflow.operators.bash_operator import BashOperator
# This makes scheduling easy
from airflow.utils.dates import days_ago

#defining DAG arguments

# You can override them on a per-task basis during operator initialization
default_args = {
    'owner': 'your_name_here',
    'start_date': days_ago(0),
    'email': ['your_email_here'],
    'retries': 1,
    'retry_delay': timedelta(minutes=5),
}

# defining the DAG

# define the DAG
dag = DAG(
    'my-first-dag',
    default_args=default_args,
    description='My first DAG',
    schedule_interval=timedelta(days=1),
)

# define the tasks

# define the first task

extract = BashOperator(
    task_id='extract',
    bash_command='cut -d":" -f1,3,6 /etc/passwd > /home/project/airflow/dags/extracted-data.txt',
    dag=dag,
)

# define the second task
transform_and_load = BashOperator(
    task_id='transform',
    bash_command='tr ":" "," < /home/project/airflow/dags/extracted-data.txt > /home/project/airflow/dags/transformed-data.csv',
    dag=dag,
)

# task pipeline
extract >> transform_and_load


In [ ]:
Exercise 4: Submit a DAG
Submitting a DAG is as simple as copying the DAG Python file into the dags folder in the AIRFLOW_HOME directory.

Airflow searches for Python source files within the specified DAGS_FOLDER. The location of DAGS_FOLDER can be located in the airflow.cfg file, where it has been configured as /home/project/airflow/dags.
    
Airflow will load the Python source files from this designated location. It will process each file, execute its contents, and subsequently load any DAG objects present in the file.

Therefore, when submitting a DAG, it is essential to position it within this directory structure. Alternatively, the AIRFLOW_HOME directory, representing the structure /home/project/airflow, can also be utilized for DAG submission.

1- Open a terminal and run the command below to set the AIRFLOW_HOME.
export AIRFLOW_HOME=/home/project/airflow
echo $AIRFLOW_HOME

2- Run the command below to submit the DAG that was created in the previous exercise.
 cp my_first_dag.py $AIRFLOW_HOME/dags

3- Verify that your DAG actually got submitted.
Run the command below to list out all the existing DAGs.
airflow dags list

4- Verify that my-first-dag is a part of the output.
airflow dags list|grep "my-first-dag"

5- Run the command below to list out all the tasks in my-first-dag. (You should see 2 tasks in the output.)
airflow tasks list my-first-dag

In [ ]:
Hands-on Lab: Monitoring a DAG
In this lab, you will work with the Airflow Web UI and CLi to explore the DAGs further. You will be exposed to using the interactive tools to search for DAGs, introduces to various views of the DAGS and how you can use this to explore the DAG workflow, the individual tasks in the workflow and view the outcome of the tasks.

# import the libraries

from datetime import timedelta
# The DAG object; we'll need this to instantiate a DAG
from airflow import DAG
# Operators; we need this to write tasks!
from airflow.operators.bash_operator import BashOperator
# This makes scheduling easy
from airflow.utils.dates import days_ago

#defining DAG arguments

# You can override them on a per-task basis during operator initialization
default_args = {
    'owner': 'Your name',
    'start_date': days_ago(0),
    'email': ['your email'],
    'retries': 1,
    'retry_delay': timedelta(minutes=5),
}

# defining the DAG
dag = DAG(
    'dummy_dag',
    default_args=default_args,
    description='My first DAG',
    schedule_interval=timedelta(minutes=1),
)

# define the tasks

# define the first task

task1 = BashOperator(
    task_id='task1',
    bash_command='sleep 1',
    dag=dag,
)

# define the second task
task2 = BashOperator(
    task_id='task2',
    bash_command='sleep 2',
    dag=dag,
)

# define the third task
task3 = BashOperator(
    task_id='task3',
    bash_command='sleep 3',
    dag=dag,
)

# task pipeline
task1 >> task2 >> task3


In [ ]:
Hands-on Lab: Working with Streaming Data using Kafka

In this lab, you will work with streaming data using Kafka. You will start by configuring the Kafka server to use the Kraft mode followed by starting the Kafka message broker service, creating a topic and then starting the producer and consumer.

In [ ]:
Exercise 1: Download and extract Kafka

1- Open a new terminal by clicking the menu bar and selecting Terminal->New Terminal, as shown in the image below.

2- Download Kafka by running the command below:
wget https://archive.apache.org/dist/kafka/3.8.0/kafka_2.13-3.8.0.tgz

3- Extract Kafka from the zip file by running the command below.
tar -xzf kafka_2.13-3.8.0.tgz


In [ ]:
Exercise 2: Configure KRaft and start server

1- Navigate to the kafka_2.13-3.8.0 directory.

cd kafka_2.13-3.8.0

2- Generate a cluster UUID that will uniquely identify the Kafka cluster.
This cluster id will be used by the KRaft controller.

KAFKA_CLUSTER_ID="$(bin/kafka-storage.sh random-uuid)"

3- KRaft requires the log directories to be configured. Run the following command to configure the log directories passing the cluster ID.

bin/kafka-storage.sh format -t $KAFKA_CLUSTER_ID -c config/kraft/server.properties

4- Now that KRaft is configured, you can start the Kafka server by running the following command.

bin/kafka-server-start.sh config/kraft/server.properties

You can be sure that the Kafka server has started when the output displays messages like "Kafka Server started

In [ ]:
Exercise 3: Create a topic and start producer
You need to create a topic before you can start to post messages.

1- Start a new terminal and change to the kafka_2.13-3.8.0 directory.
cd kafka_2.13-3.8.0

2- To create a topic named news, run the command below.
bin/kafka-topics.sh --create --topic news --bootstrap-server localhost:9092
You will see the message: Created topic news.

3- You need a producer to send messages to Kafka. Run the command below to start a producer.
bin/kafka-console-producer.sh   --bootstrap-server localhost:9092   --topic news

4- After the producer starts, and you get the '>' prompt, type any text message and press enter. Or you can copy the text below and paste. The below text sends three messages to Kafka.
    
Good morning
Good day
Enjoy the Kafka lab

In [ ]:
Exercise 4: Start Consumer
You need a consumer to read messages from Kafka.

1- Start a new terminal and change to the kafka_2.13-3.8.0 directory.
cd kafka_2.13-3.8.0

2- Run the command below to listen to the messages in the topic news.
bin/kafka-console-consumer.sh   --bootstrap-server localhost:9092   --topic news   --from-beginning

3- You should see all the messages you sent from the producer appear here.

4- You can go back to the producer terminal and type some more messages, one message per line, and you will see them appear here.
    


In [ ]:
Exercise 5: Explore Kafka directories
Kafka uses the /tmp//tmp/kraft-combined-logs directory to store the messages.

1- Start a new terminal and navigate to the kafka_2.13-3.8.0 directory.
cd kafka_2.13-3.8.0

2- Explore the root directory of the server.
ls

3- Notice there is a tmp directory. The kraft-combine-logs inside the tmp directory contains all the logs. To check the logs generated for the topic news run the following command:
ls /tmp/kraft-combined-logs/news-0

In [ ]:
Exercise 6: Clean up

To stop the producer
In the terminal where you are running producer, press CTRL+C.

To stop the consumer
In the terminal where you are running consumer, press CTRL+C.

To stop the Kafka server
In the terminal where you are running Kafka server, press CTRL+C.